# Member 2 — Regression Models, Evaluation, Tuning, CV, Diagnostics

Input: preprocessed train/test split from Member 1 (`data/processed/regression_train.csv`, `regression_test.csv`).
Scope: 10 regression algorithms, comparison table, GridSearchCV on 2 best models, 5-fold CV, residual/pred-vs-actual
and feature-importance diagnostics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, root_mean_squared_error

RANDOM_STATE = 42
plt.style.use('seaborn-v0_8')
sns.set_palette('tab10')

## 1. Load preprocessed split (owned by Member 1)

Dataset: `data/airquality/AirQualityUCI.csv` (UCI Air Quality). Target: `C6H6(GT)` (benzene concentration —
only ~4% missing vs. 90% for `NMHC(GT)`, so it's the usable regression target). `-200` is the dataset's
missing-value sentinel.

Tries Member 1's cleaned/split CSVs first; falls back to a raw loader with a placeholder impute so this
notebook is not blocked on their output landing.

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split

TARGET = 'C6H6(GT)'
PROCESSED_TRAIN = Path('../data/processed/regression_train.csv')
PROCESSED_TEST = Path('../data/processed/regression_test.csv')

if PROCESSED_TRAIN.exists() and PROCESSED_TEST.exists():
    train_df = pd.read_csv(PROCESSED_TRAIN)
    test_df = pd.read_csv(PROCESSED_TEST)
else:
    print('data/processed split not found — using raw-loader fallback')
    raw = pd.read_csv('../data/airquality/AirQualityUCI.csv', sep=';', decimal=',')
    raw = raw.loc[:, ~raw.columns.str.contains('^Unnamed')].dropna(how='all')
    raw = raw.drop(columns=['Date', 'Time', 'NMHC(GT)'])  # NMHC(GT) ~90% missing, unusable
    raw = raw.replace(-200, np.nan)
    raw = raw.dropna(subset=[TARGET])  # can't train/eval without target
    raw = raw.fillna(raw.median(numeric_only=True))  # placeholder impute; Member 1 owns final strategy
    train_df, test_df = train_test_split(raw, test_size=0.2, random_state=RANDOM_STATE)

X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]
X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET]
X_train.shape, X_test.shape

## 2. Define all 10 models

Built up incrementally, one algorithm family per commit.

In [ ]:
models = {}

In [ ]:
models.update({
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(random_state=RANDOM_STATE),
    'Lasso Regression': Lasso(random_state=RANDOM_STATE),
    'ElasticNet Regression': ElasticNet(random_state=RANDOM_STATE),
})
list(models.keys())

In [ ]:
models['Polynomial Regression'] = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('lin', LinearRegression())
])
list(models.keys())

In [ ]:
models['Decision Tree Regressor'] = DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)
list(models.keys())